In [1]:
import os, psutil, gc
import time 
import json
import pprint

from collections import defaultdict
import random
import numpy as np

import torch 
import torch.distributed as dist
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"

# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [4]:


# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 20            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.date_string = "Aug 1 2025"
config.seed = 0

# mcts parameter
config.num_batches = 20
config.step_budget = config.num_batches*config.max_depths 
config.num_phases = 1000

config.lam = 0.01 
config.normalize_embeds = True
config.use_ppl = True

config.ds_beta = 1.0
config.ds_alpha = 100.0
config.negative_reward = 0

config.version = "q71"


# baseline: gpu_memory_utilization=0.2
# use the standard model 
llm_total_gpu = 0.4
llm_gpu_memory_utilization = 0.2

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=config.seed,
)



INFO 03-11 15:57:38 [utils.py:238] non-default args: {'dtype': 'float16', 'max_model_len': 5000, 'swap_space': 16, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct'}
INFO 03-11 15:57:54 [model.py:531] Resolved architecture: LlamaForCausalLM
WARNING 03-11 15:57:54 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 03-11 15:57:54 [model.py:1554] Using max model len 5000
INFO 03-11 15:57:54 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-11 15:57:54 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-11 15:57:54 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-11 15:57:54 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-11 15:5

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=710640) INFO 03-11 15:58:00 [default_loader.py:293] Loading weights took 1.35 seconds
(EngineCore_DP0 pid=710640) INFO 03-11 15:58:00 [gpu_model_runner.py:4364] Model loading took 2.32 GiB memory and 3.401240 seconds
(EngineCore_DP0 pid=710640) INFO 03-11 15:58:40 [gpu_worker.py:424] Available KV cache memory: 3.47 GiB
(EngineCore_DP0 pid=710640) INFO 03-11 15:58:40 [kv_cache_utils.py:1314] GPU KV cache size: 113,664 tokens
(EngineCore_DP0 pid=710640) INFO 03-11 15:58:40 [kv_cache_utils.py:1319] Maximum concurrency for 5,000 tokens per request: 22.70x
(EngineCore_DP0 pid=710640) INFO 03-11 15:58:40 [core.py:282] init engine (profile, create kv cache, warmup model) took 39.77 seconds
(EngineCore_DP0 pid=710640) (EngineCore_DP0 pid=710640) WARNING 03-11 15:58:41 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-11 15:58:41 [vllm.py:792] Inductor compilation was disabl

TypeError: EngineArgs.__init__() got an unexpected keyword argument 'task'

ERROR 03-11 16:16:09 [core_client.py:691] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.


In [5]:
llm_vllm_embeds = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    # task="embed",
    runner="pooling",
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=config.seed,
)

INFO 03-11 15:59:52 [utils.py:238] non-default args: {'runner': 'pooling', 'dtype': 'float16', 'max_model_len': 5000, 'swap_space': 16, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct'}
INFO 03-11 15:59:52 [model.py:856] Resolved `--convert auto` to `--convert embed`. Pass the value explicitly to silence this message.
INFO 03-11 15:59:52 [model.py:531] Resolved architecture: LlamaForCausalLM
WARNING 03-11 15:59:52 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 03-11 15:59:52 [model.py:1554] Using max model len 5000
INFO 03-11 15:59:52 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 03-11 15:59:52 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-11 15:59:52 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations setti

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=711049) INFO 03-11 15:59:56 [default_loader.py:293] Loading weights took 1.33 seconds
(EngineCore_DP0 pid=711049) INFO 03-11 15:59:57 [gpu_model_runner.py:4364] Model loading took 2.32 GiB memory and 2.110967 seconds
(EngineCore_DP0 pid=711049) INFO 03-11 15:59:58 [gpu_worker.py:424] Available KV cache memory: 3.52 GiB
(EngineCore_DP0 pid=711049) INFO 03-11 15:59:58 [kv_cache_utils.py:1314] GPU KV cache size: 115,344 tokens
(EngineCore_DP0 pid=711049) INFO 03-11 15:59:58 [kv_cache_utils.py:1319] Maximum concurrency for 5,000 tokens per request: 23.03x
(EngineCore_DP0 pid=711049) INFO 03-11 15:59:58 [core.py:282] init engine (profile, create kv cache, warmup model) took 1.61 seconds
(EngineCore_DP0 pid=711049) WARNING 03-11 15:59:59 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore_DP0 pid=711049) WARNING 03-11 15:59:59 [vllm.py:792] Inductor compilation was disable

In [6]:
# pooling_params = PoolingParams(strategy="ALL")
outputs = llm_vllm_embeds.encode("Extract average embeddings for this sentence.", pooling_task="token_embed", use_tqdm=False)

In [8]:
print(len(outputs))

1


In [10]:
tokens_embeds = outputs[0].outputs.data
avg_embed = tokens_embeds.mean(dim=0) 
last_embed = tokens_embeds[-1]
print(avg_embed.shape)
print(avg_embed[:5])
print(last_embed[:5])
# mean_embedding = np.mean(outputs_embeds, axis=0)

torch.Size([2048])
tensor([-0.0143,  0.0306,  0.0215, -0.0022,  0.0229])
tensor([-0.0305,  0.0348,  0.0120,  0.0050,  0.0505])


In [ ]:
mean_embeddings = []
last_token_embeddings = []

for output in outputs:
    # shape: [seq_len, hidden_dim]
    token_embeds = output.outputs.data.detach().clone()

    # Average over all prompt tokens
    mean_embed = token_embeds.mean(dim=0)

    # Last token embedding
    last_embed = token_embeds[-1]

    # Optional normalization, depending on your downstream use
    mean_embed = torch.nn.functional.normalize(mean_embed, dim=-1)
    last_embed = torch.nn.functional.normalize(last_embed, dim=-1)

    mean_embeddings.append(mean_embed)
    last_token_embeddings.append(last_embed)

print(mean_embeddings)       # [hidden_dim]
print(last_token_embeddings) # [hidden_dim]